In [ ]:
import json
import random
import re
import openai
from typing import List, Dict
import os
import sys
from tqdm import tqdm

# 设置OpenAI API密钥
openai.api_key = 'YOUR_OPENAI_API_KEY'  # 请替换为您的OpenAI API密钥

# 定义实体类型映射
ENTITY_TYPE_MAPPING = {
    "Nh": "Person",
    "Ns": "Location",
    "NT": "Time",
    "NDR": "DrugType",
    "NW": "DrugWeight"
}

# 定义可用实体类型
ENTITY_TYPES = list(ENTITY_TYPE_MAPPING.keys())

def load_dataset(file_path: str) -> List[Dict]:
    """
    从指定的JSON Lines文件中加载数据集。

    Args:
        file_path (str): JSON Lines文件的路径。

    Returns:
        List[Dict]: 数据集列表。
    """
    dataset = []
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    data = json.loads(line)
                    dataset.append(data)
        return dataset
    except Exception as e:
        print(f"加载数据集时出错: {e}")
        sys.exit(1)

def format_entity(entity_text: str, entity_type: str) -> str:
    """
    格式化实体为 <Type>("Entity") 的形式。

    Args:
        entity_text (str): 实体文本。
        entity_type (str): 实体类型。

    Returns:
        str: 格式化后的实体字符串。
    """
    return f'<{entity_type}>("{entity_text}")'

def prepare_few_shot_examples(dataset: List[Dict], num_examples: int = 5) -> str:
    """
    从数据集中随机选择num_examples个示例，并格式化为提示。

    Args:
        dataset (List[Dict]): 数据集列表。
        num_examples (int, optional): 要选择的示例数量。默认值为5。

    Returns:
        str: 格式化后的Few-Shot示例字符串。
    """
    if len(dataset) < num_examples:
        examples = dataset
    else:
        examples = random.sample(dataset, num_examples)
    
    formatted_examples = []
    for example in examples:
        sent = example.get('sentText', '')
        for mention in example.get('entityMentions', []):
            entity_type = ENTITY_TYPE_MAPPING.get(mention['label'], 'Unknown')
            entity_text = mention['text']
            # 使用正则确保只替换完整的实体
            sent = re.sub(r'(?<!\w)' + re.escape(entity_text) + r'(?!\w)', format_entity(entity_text, entity_type), sent)
        formatted_examples.append(sent)
    return "\n".join(formatted_examples)

def select_random_entities(entity_mentions: List[Dict]) -> List[Dict]:
    """
    随机选择n个实体，并随机分配实体类型。

    Args:
        entity_mentions (List[Dict]): 实体列表。

    Returns:
        List[Dict]: 选定的实体列表，包含重新分配的实体类型。
    """
    N = len(entity_mentions)
    if N == 0:
        return []
    n = random.randint(3, N)  # n的范围是0到N
    selected_entities = random.sample(entity_mentions, n) if n > 0 else []
    for entity in selected_entities:
        # 随机选择一个实体类型
        entity['assigned_label'] = random.choice(ENTITY_TYPES)
    return selected_entities

def generate_prompt(selected_entities: List[Dict], few_shot: str) -> str:
    """
    生成给LLM的提示，包括few-shot示例和要生成的句子实体。

    Args:
        selected_entities (List[Dict]): 选定的实体列表。
        few_shot (str): Few-Shot示例字符串。

    Returns:
        str: 完整的提示字符串。
    """
    if selected_entities:
        entities_formatted = ", ".join([
            format_entity(ent['text'], ENTITY_TYPE_MAPPING.get(ent['assigned_label'], 'Unknown'))
            for ent in selected_entities
        ])
    else:
        entities_formatted = "无实体"  # 如果没有选定实体，可以根据需要调整
    prompt = (
        f"{few_shot}\n\n"
        "请根据以下实体生成一个包含这些实体的句子：\n"
        f"{entities_formatted}\n"
        "句子："
    )
    return prompt

def call_openai(prompt: str, max_tokens: int = 150) -> str:
    """
    调用OpenAI的API生成句子。

    Args:
        prompt (str): 提示文本。
        max_tokens (int, optional): 生成文本的最大token数。默认值为150。

    Returns:
        str: 生成的句子。
    """
    try:
        response = openai.Completion.create(
            engine="gpt-4",  # 或者使用具体的GPT-4型号，如 "gpt-4-0613"
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=0.7,
            n=1,
            stop=None
        )
        generated_text = response.choices[0].text.strip()
        return generated_text
    except Exception as e:
        print(f"调用OpenAI API时出错: {e}")
        return ""

def reannotate_entities(generated_sentence: str, selected_entities: List[Dict]) -> str:
    """
    在生成的句子中重新标注实体。

    Args:
        generated_sentence (str): 生成的句子。
        selected_entities (List[Dict]): 选定的实体列表。

    Returns:
        str: 重新标注后的句子。
    """
    for ent in selected_entities:
        entity_text = ent['text']
        entity_type = ENTITY_TYPE_MAPPING.get(ent['assigned_label'], 'Unknown')
        # 使用正则表达式进行全局替换，避免部分匹配
        pattern = re.escape(entity_text)
        replacement = format_entity(entity_text, entity_type)
        # 仅替换首次出现，可以根据需求调整
        generated_sentence = re.sub(pattern, replacement, generated_sentence, count=1)
    return generated_sentence

def validate_sentence(sentence: str) -> bool:
    """
    验证句子是否符合模板：
    - 不包含换行符
    - 仅包含一个句子（以中文句号或英文句号结尾）

    Args:
        sentence (str): 需要验证的句子。

    Returns:
        bool: 如果符合要求，返回True；否则，返回False。
    """
    if "\n" in sentence:
        return False
    # 判断句子是否以句号或英文句号结尾
    if not re.match(r'.*[。.]$', sentence):
        return False
    # 检查是否只有一个句号或英文句号
    if sentence.count('。') + sentence.count('.') > 1:
        return False
    return True

def generate_sentence_for_data_point(data: Dict, dataset: List[Dict]) -> str:
    """
    为单个数据项生成符合要求的句子。

    Args:
        data (Dict): 单个数据项。
        dataset (List[Dict]): 数据集列表，用于准备Few-Shot示例。

    Returns:
        str: 生成的句子，如果生成失败或不符合要求，则返回空字符串。
    """
    entity_mentions = data.get('entityMentions', [])
    selected_entities = select_random_entities(entity_mentions)
    few_shot = prepare_few_shot_examples(dataset, num_examples=5)
    prompt = generate_prompt(selected_entities, few_shot)
    print(selected_entities)
    # generated = call_openai(prompt)
    # if not generated:
    #     return ""
    # if not validate_sentence(generated):
    #     return ""
    # # 重新标注实体
    # reannotated = reannotate_entities(generated, selected_entities)
    # return reannotated

def save_generated_sentences(output_path: str, generated_data: List[Dict]):
    """
    将生成的句子保存到指定的JSON文件中。

    Args:
        output_path (str): 输出文件的路径。
        generated_data (List[Dict]): 生成的数据列表。
    """
    try:
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(generated_data, f, ensure_ascii=False, indent=4)
        print(f"生成的句子已保存到 {output_path}")
    except Exception as e:
        print(f"保存生成的句子时出错: {e}")

def main(train_file: str, output_file: str):
    """
    主函数，执行生成句子的全过程。

    Args:
        train_file (str): 训练数据文件的路径。
        output_file (str): 输出文件的路径。
    """
    print("加载数据集...")
    dataset = load_dataset(train_file)
    print(f"数据集中包含 {len(dataset)} 条数据。")

    generated_data = []
    print("开始生成句子...")
    for data in tqdm(dataset, desc="生成句子"):
        generated_sentence = generate_sentence_for_data_point(data, dataset)
    #     if generated_sentence:
    #         generated_data.append({
    #             "originalArticleId": data.get("articleId"),
    #             "originalSentId": data.get("sentId"),
    #             "generatedSentence": generated_sentence
    #         })
    
    # print(f"成功生成 {len(generated_data)} 个句子。")
    # save_generated_sentences(output_file, generated_data)

if __name__ == "__main__":
    # 定义输入和输出文件路径
    TRAIN_FILE = "train.json"  # 输入的训练数据文件
    OUTPUT_FILE = "generated_sentences.json"  # 输出的生成句子文件

    # 检查输入文件是否存在
    if not os.path.exists(TRAIN_FILE):
        print(f"训练数据文件 {TRAIN_FILE} 不存在。请确保文件存在于当前目录。")
        sys.exit(1)

    # 运行主函数
    main(TRAIN_FILE, OUTPUT_FILE)

加载数据集...
数据集中包含 1415 条数据。
开始生成句子...


生成句子:   8%|▊         | 107/1415 [00:00<00:01, 1064.47it/s]

[{'end': 90, 'start': 87, 'text': '林某某', 'label': 'Nh', 'assigned_label': 'Ns'}, {'end': 209, 'start': 206, 'text': '海洛因', 'label': 'NDR', 'assigned_label': 'NT'}, {'end': 76, 'start': 63, 'text': '2014年8月7日早上7时', 'label': 'NT', 'assigned_label': 'Ns'}, {'end': 202, 'start': 197, 'text': '0.44克', 'label': 'NW', 'assigned_label': 'NDR'}, {'end': 146, 'start': 143, 'text': '林某某', 'label': 'Nh', 'assigned_label': 'Nh'}, {'end': 187, 'start': 182, 'text': '0.07克', 'label': 'NW', 'assigned_label': 'Nh'}]
[{'end': 86, 'start': 83, 'text': '焦某某', 'label': 'Nh', 'assigned_label': 'Ns'}, {'end': 40, 'start': 34, 'text': '贵阳市云岩区', 'label': 'Ns', 'assigned_label': 'NDR'}, {'end': 18, 'start': 6, 'text': '2015年7月22日1时', 'label': 'NT', 'assigned_label': 'NW'}, {'end': 115, 'start': 111, 'text': '120克', 'label': 'NW', 'assigned_label': 'NW'}, {'end': 31, 'start': 28, 'text': '吴某某', 'label': 'Nh', 'assigned_label': 'NDR'}, {'end': 109, 'start': 106, 'text': '海洛因', 'label': 'NDR', 'assigned_label': 'N

生成句子:  16%|█▌        | 220/1415 [00:00<00:01, 1098.64it/s]

[{'end': 266, 'start': 264, 'text': '何某', 'label': 'Nh', 'assigned_label': 'Ns'}, {'end': 82, 'start': 80, 'text': '何某', 'label': 'Nh', 'assigned_label': 'NDR'}, {'end': 163, 'start': 156, 'text': '1.8335克', 'label': 'NW', 'assigned_label': 'NT'}, {'end': 193, 'start': 190, 'text': '唐某某', 'label': 'Nh', 'assigned_label': 'Ns'}, {'end': 189, 'start': 186, 'text': '焦某某', 'label': 'Nh', 'assigned_label': 'NT'}]
[{'end': 119, 'start': 117, 'text': '冰毒', 'label': 'NDR', 'assigned_label': 'Ns'}, {'end': 20, 'start': 7, 'text': '2013年12月6日16时', 'label': 'NT', 'assigned_label': 'Ns'}, {'end': 71, 'start': 68, 'text': '廖某某', 'label': 'Nh', 'assigned_label': 'Ns'}, {'end': 129, 'start': 125, 'text': '1.2克', 'label': 'NW', 'assigned_label': 'Nh'}]
[{'end': 89, 'start': 87, 'text': '蓝某', 'label': 'Nh', 'assigned_label': 'Nh'}, {'end': 32, 'start': 27, 'text': '5月14日', 'label': 'NT', 'assigned_label': 'NT'}, {'end': 86, 'start': 77, 'text': '上林县乔贤镇龙华街', 'label': 'Ns', 'assigned_label': 'NW'}, {'end

...

[{'end': 95, 'start': 93, 'text': '王某', 'label': 'Nh', 'assigned_label': 'NDR'}, {'end': 23, 'start': 11, 'text': '2014年1月18日中午', 'label': 'NT', 'assigned_label': 'Ns'}, {'end': 116, 'start': 113, 'text': '戎某甲', 'label': 'Nh', 'assigned_label': 'NW'}, {'end': 92, 'start': 90, 'text': '冰毒', 'label': 'NDR', 'assigned_label': 'NT'}, {'end': 140, 'start': 138, 'text': '当晚', 'label': 'NT', 'assigned_label': 'NT'}, {'end': 190, 'start': 188, 'text': '陈某', 'label': 'Nh', 'assigned_label': 'Nh'}, {'end': 29, 'start': 27, 'text': '陈某', 'label': 'Nh', 'assigned_label': 'Ns'}, {'end': 119, 'start': 117, 'text': '王某', 'label': 'Nh', 'assigned_label': 'Ns'}, {'end': 62, 'start': 60, 'text': '陈某', 'label': 'Nh', 'assigned_label': 'NT'}]
[{'end': 25, 'start': 21, 'text': '12月份', 'label': 'NT', 'assigned_label': 'NW'}, {'end': 30, 'start': 28, 'text': '郭某', 'label': 'Nh', 'assigned_label': 'NDR'}, {'end': 20, 'start': 11, 'text': '2014年10月份', 'label': 'NT', 'assigned_label': 'NW'}]
[{'end': 101, 'star

生成句子:  91%|█████████▏| 1293/1415 [00:01<00:00, 1067.67it/s]

[{'end': 82, 'start': 76, 'text': '36.94克', 'label': 'NW', 'assigned_label': 'NT'}, {'end': 72, 'start': 67, 'text': '甲基苯丙胺', 'label': 'NDR', 'assigned_label': 'Ns'}, {'end': 33, 'start': 21, 'text': '2014年9月4日20时', 'label': 'NT', 'assigned_label': 'NDR'}, {'end': 75, 'start': 73, 'text': '冰毒', 'label': 'NDR', 'assigned_label': 'Ns'}, {'end': 44, 'start': 38, 'text': '崇安区新开河', 'label': 'Ns', 'assigned_label': 'Ns'}]
[{'end': 30, 'start': 28, 'text': '徐某', 'label': 'Nh', 'assigned_label': 'Ns'}, {'end': 23, 'start': 7, 'text': '2016年3月28日19时40分', 'label': 'NT', 'assigned_label': 'Nh'}, {'end': 113, 'start': 111, 'text': '冰毒', 'label': 'NDR', 'assigned_label': 'Ns'}, {'end': 57, 'start': 31, 'text': '六盘水市钟山区民族路“XXXX时尚酒店”对面小区门口', 'label': 'Ns', 'assigned_label': 'NT'}, {'end': 175, 'start': 170, 'text': '甲基苯丙胺', 'label': 'NDR', 'assigned_label': 'NW'}, {'end': 78, 'start': 73, 'text': '甲基苯丙胺', 'label': 'NDR', 'assigned_label': 'NW'}, {'end': 110, 'start': 105, 'text': '甲基苯丙胺', 'label': 'N

生成句子: 100%|██████████| 1415/1415 [00:01<00:00, 1039.05it/s]

[{'end': 148, 'start': 146, 'text': '杨某', 'label': 'Nh', 'assigned_label': 'Ns'}, {'end': 118, 'start': 105, 'text': '2015年10月9日22时', 'label': 'NT', 'assigned_label': 'Ns'}, {'end': 52, 'start': 38, 'text': '东莞市蛟乙塘市场附近一出租屋', 'label': 'Ns', 'assigned_label': 'Nh'}, {'end': 169, 'start': 153, 'text': '2015年10月10日1时30分', 'label': 'NT', 'assigned_label': 'NDR'}, {'end': 25, 'start': 15, 'text': '2015年10月1日', 'label': 'NT', 'assigned_label': 'NDR'}, {'end': 70, 'start': 68, 'text': '陈某', 'label': 'Nh', 'assigned_label': 'NW'}, {'end': 88, 'start': 85, 'text': '田某乙', 'label': 'Nh', 'assigned_label': 'Nh'}, {'end': 122, 'start': 120, 'text': '陈某', 'label': 'Nh', 'assigned_label': 'Ns'}, {'end': 91, 'start': 89, 'text': '杨某', 'label': 'Nh', 'assigned_label': 'NW'}, {'end': 66, 'start': 53, 'text': '2015年10月7日23时', 'label': 'NT', 'assigned_label': 'NW'}, {'end': 145, 'start': 143, 'text': '肖某', 'label': 'Nh', 'assigned_label': 'NW'}, {'end': 33, 'start': 31, 'text': '陈某', 'label': 'Nh', 'assign

In [11]:
import json
dataset=[]
with open('train.json', 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            data = json.loads(line)
            dataset.append(data['entityMentions'])
dataset

[[{'end': 20, 'start': 17, 'text': '林某某', 'label': 'Nh'},
  {'end': 51, 'start': 48, 'text': '海洛因', 'label': 'NDR'},
  {'end': 76, 'start': 63, 'text': '2014年8月7日早上7时', 'label': 'NT'},
  {'end': 85, 'start': 82, 'text': '徐某某', 'label': 'Nh'},
  {'end': 90, 'start': 87, 'text': '林某某', 'label': 'Nh'},
  {'end': 123, 'start': 120, 'text': '徐某某', 'label': 'Nh'},
  {'end': 146, 'start': 143, 'text': '林某某', 'label': 'Nh'},
  {'end': 187, 'start': 182, 'text': '0.07克', 'label': 'NW'},
  {'end': 202, 'start': 197, 'text': '0.44克', 'label': 'NW'},
  {'end': 209, 'start': 206, 'text': '海洛因', 'label': 'NDR'}],
 [{'end': 18, 'start': 6, 'text': '2015年7月22日1时', 'label': 'NT'},
  {'end': 31, 'start': 28, 'text': '吴某某', 'label': 'Nh'},
  {'end': 40, 'start': 34, 'text': '贵阳市云岩区', 'label': 'Ns'},
  {'end': 86, 'start': 83, 'text': '焦某某', 'label': 'Nh'},
  {'end': 109, 'start': 106, 'text': '海洛因', 'label': 'NDR'},
  {'end': 115, 'start': 111, 'text': '120克', 'label': 'NW'}],
 [{'end': 15, 'start': 7, '

In [12]:
# 遍历数据结构，去掉每个字典中的 'end' 和 'start' 键
for sublist in dataset:
    for item in sublist:
        item.pop('end', None)  # 删除 'end' 键
        item.pop('start', None)  # 删除 'start' 键

print(dataset)

[[{'text': '林某某', 'label': 'Nh'}, {'text': '海洛因', 'label': 'NDR'}, {'text': '2014年8月7日早上7时', 'label': 'NT'}, {'text': '徐某某', 'label': 'Nh'}, {'text': '林某某', 'label': 'Nh'}, {'text': '徐某某', 'label': 'Nh'}, {'text': '林某某', 'label': 'Nh'}, {'text': '0.07克', 'label': 'NW'}, {'text': '0.44克', 'label': 'NW'}, {'text': '海洛因', 'label': 'NDR'}], [{'text': '2015年7月22日1时', 'label': 'NT'}, {'text': '吴某某', 'label': 'Nh'}, {'text': '贵阳市云岩区', 'label': 'Ns'}, {'text': '焦某某', 'label': 'Nh'}, {'text': '海洛因', 'label': 'NDR'}, {'text': '120克', 'label': 'NW'}], [{'text': '2014年12月', 'label': 'NT'}, {'text': '黄某某', 'label': 'Nh'}, {'text': '博罗县罗阳镇田牌村', 'label': 'Ns'}, {'text': '氯胺酮', 'label': 'NDR'}, {'text': '黄某明', 'label': 'Nh'}, {'text': '刘某林', 'label': 'Nh'}, {'text': '刘某强', 'label': 'Nh'}, {'text': '2015年2月份', 'label': 'NT'}, {'text': '黄某某', 'label': 'Nh'}, {'text': '博罗县罗阳镇田牌村', 'label': 'Ns'}, {'text': '甲基苯丙胺', 'label': 'NDR'}, {'text': '冰毒', 'label': 'NDR'}, {'text': '胡某康', 'label': 'Nh'}, {'text': '

In [14]:
with open('entity.json','w') as file:
    json.dump(dataset, file, ensure_ascii=False)


In [15]:
categorized_data = {}

# 遍历数据进行分类
for sublist in dataset:
    for item in sublist:
        label = item['label']
        if label not in categorized_data:
            categorized_data[label] = []
        categorized_data[label].append(item)

# 将归类后的数据保存为 JSON 文件
with open('categorized_data.json', 'w', encoding='utf-8') as f:
    json.dump(categorized_data, f, ensure_ascii=False)

print("Data has been categorized and saved to 'categorized_data.json'")

Data has been categorized and saved to 'categorized_data.json'


### 随机抽取实体和实体类型

In [6]:
import json
import random

# 假设已经从文件中加载了 JSON 数据，或者直接用一个字典变量 data 来代替
with open('categorized_data.json', 'r', encoding='utf-8') as f:
    categorized_data = json.load(f)

# 定义需要采样的实体类型
required_labels = ['Nh', 'NDR']
optional_labels = ['Ns', 'NT', 'NW']

# 结果存储字典
sampled_data = {}

# 随机采样函数：采样指定实体类型的 1 到 3 个实体
def sample_entities(label, max_samples=3):
    entities = categorized_data.get(label, [])
    sample_count = random.randint(1, max_samples)  # 随机选 1 到 3 个实体
    sampled_entities = random.sample(entities, min(sample_count, len(entities)))
    return [entity['text'] for entity in sampled_entities]

# 首先确保包含 Nh 和 NDR
sampled_data['Nh'] = sample_entities('Nh', max_samples=3)
sampled_data['NDR'] = sample_entities('NDR', max_samples=3)

# 随机选择其他实体类型
chosen_optional_labels = random.sample(optional_labels, random.randint(1, len(optional_labels)))

# 采样其他实体类型
for label in chosen_optional_labels:
    sampled_data[label] = sample_entities(label, max_samples=3)

key_mapping = {
    'Nh': '人名',
    'NDR': '毒品',
    'NT': '时间',
    'Ns': '地名',
    'NW': '毒品重量'
}

mapped_data = {key_mapping.get(k, k): v for k, v in sampled_data.items()}
mapped_data

{'人名': ['梁某某'],
 '毒品': ['冰毒'],
 '毒品重量': ['1.25克', '1.29克', '31.19克'],
 '地名': ['上海市浦东新区川沙新镇中医医院', '八一宾馆', '本市洪山区绿岛新区4栋3门某某的家中'],
 '时间': ['2014年1月11日9时', '2015年5月24日11时', '2014年12月1日12时']}

## 随机抽取句子

In [7]:
import json
import random

# 要搜索的实体类型
entity_types_to_search = list(sampled_data.keys())

# 结果存储
matching_sentences = []

# 逐行读取 `train.json` 并搜索匹配的句子
with open('train.json', 'r', encoding='utf-8') as f:
    lines = f.readlines()  # 读取所有行
    random.shuffle(lines)  # 打乱行的顺序
    for line in lines:
        # 解析每一行 JSON 数据
        data = json.loads(line)
        sent_text = data.get("sentText", "")
        entity_mentions = data.get("entityMentions", [])
        
        # 提取该句子中的所有实体的文本和标签
        entity_texts = {entity['text'] for entity in entity_mentions}
        
        # 检查采样实体是否出现在这个句子中
        match_found = False
        for entity_type in entity_types_to_search:
            # 如果该类型有实体并且该实体在句子中出现，则匹配
            for entity in sampled_data[entity_type]:
                if entity in entity_texts:
                    match_found = True
                    break
            if match_found:
                break
        
        # 如果匹配，保存该句子
        if match_found:
            matching_sentences.append(sent_text)
        
        # 如果已找到5个句子，停止搜索
        if len(matching_sentences) >= 3:
            break

# 输出结果，包含符合条件的 5 个句子
print(len(matching_sentences),matching_sentences)


3 ['经审理查明，2013年5月初的一天晚上，被告人某某某驾驶渝某某某号小轿车至铜梁县君悦大豪园小区门口公路边，将一小包毒品甲基苯丙胺（冰毒）以200元钱的价格卖给某某。', '亳州市谯城区人民检察院指控：2014年12月10日凌晨，被告人葛某某在其住宿的本市谯城区一帆风顺快捷酒店419房间内，提供甲基苯丙胺（冰毒）和吸食甲基苯丙胺（冰毒）用的冰壶，并容留陈伟、苏珍、王超三人在419房间内吸食甲基苯丙胺（冰毒）。', '广汉市人民检察院指控：2013年11月8日9时许，广汉市公安局民警依法对被告人谭某某位于广汉市雒城镇中山大道北二段110号2单元302室的出租房进行搜查。民警从被告人谭某某的卧室内查获净重为0.4克的疑似毒品“冰毒”一小袋、净重为45.28克的疑似毒品“冰毒”一大袋、AMPUT黑壳电子秤一个、印有“冰川时代”字样的塑料制作的吸毒工具“壶壶”一个、塑料分装带若干、锡箔纸若干。经德阳市公安局物证鉴定室鉴定，从所查获的疑似毒品冰毒中均检出甲基苯丙胺成分。']


In [14]:
def generate_prompt(entities, example_sentences):
    prompt = "根据以下提供的实体类型和具体实体，生成一个包含相关内容的句子，类似于给定的样本句子风格。生成的句子需要真实且符合上下文逻辑，并包括以下实体类型中至少 3 种的内容：\n\n"
    
    # 添加实体类型和实体数据
    prompt += "### 实体类型与实体：\n"
    for entity_type, values in entities.items():
        prompt += f"- **{entity_type}**：{', '.join(values)}\n"
    
    # 添加示例句子
    prompt += "\n### 示例句子风格：\n"
    for sentence in example_sentences:
        prompt += f"- \"{sentence}\"\n"
    
    prompt += "\n### 生成要求：\n请生成一个句子，要求逻辑清晰且包含以下实体类型中的人名和毒品实体。并且请按照这种格式输出{\"entityMentions\": [{\"text\": \"林某某\", \"label\": \"人名\"}, {\"text\": \"海洛因\", \"label\": \"毒品\"}], \"sentText\": \"经查，林某某因贩卖海洛因被警方逮捕\"}]}\n"
    return prompt

# 生成 Prompt
prompt_text = generate_prompt(mapped_data, matching_sentences)
prompt_text

'根据以下提供的实体类型和具体实体，生成一个包含相关内容的句子，类似于给定的样本句子风格。生成的句子需要真实且符合上下文逻辑，并包括以下实体类型中至少 3 种的内容：\n\n### 实体类型与实体：\n- **人名**：梁某某\n- **毒品**：冰毒\n- **毒品重量**：1.25克, 1.29克, 31.19克\n- **地名**：上海市浦东新区川沙新镇中医医院, 八一宾馆, 本市洪山区绿岛新区4栋3门某某的家中\n- **时间**：2014年1月11日9时, 2015年5月24日11时, 2014年12月1日12时\n\n### 示例句子风格：\n- "经审理查明，2013年5月初的一天晚上，被告人某某某驾驶渝某某某号小轿车至铜梁县君悦大豪园小区门口公路边，将一小包毒品甲基苯丙胺（冰毒）以200元钱的价格卖给某某。"\n- "亳州市谯城区人民检察院指控：2014年12月10日凌晨，被告人葛某某在其住宿的本市谯城区一帆风顺快捷酒店419房间内，提供甲基苯丙胺（冰毒）和吸食甲基苯丙胺（冰毒）用的冰壶，并容留陈伟、苏珍、王超三人在419房间内吸食甲基苯丙胺（冰毒）。"\n- "广汉市人民检察院指控：2013年11月8日9时许，广汉市公安局民警依法对被告人谭某某位于广汉市雒城镇中山大道北二段110号2单元302室的出租房进行搜查。民警从被告人谭某某的卧室内查获净重为0.4克的疑似毒品“冰毒”一小袋、净重为45.28克的疑似毒品“冰毒”一大袋、AMPUT黑壳电子秤一个、印有“冰川时代”字样的塑料制作的吸毒工具“壶壶”一个、塑料分装带若干、锡箔纸若干。经德阳市公安局物证鉴定室鉴定，从所查获的疑似毒品冰毒中均检出甲基苯丙胺成分。"\n\n### 生成要求：\n请生成一个句子，要求逻辑清晰且包含以下实体类型中的人名和毒品实体。并且请按照这种格式输出{"entityMentions": [{"text": "林某某", "label": "人名"}, {"text": "海洛因", "label": "毒品"}], "sentText": "经查，林某某因贩卖海洛因被警方逮捕"}]}\n'

In [16]:
from zhipuai import ZhipuAI
client = ZhipuAI(api_key="7bb97f4973df63dbb95018423009ed65.g4p3Q4YfZgdtOm4s")  # 请填写您自己的APIKey
response = client.chat.completions.create(
    model="glm-4-flash",  # 请填写您要调用的模型名称
    messages=[
        {"role": "user", "content": prompt_text},
    ],
)
print(response.choices[0].message.content)

{"entityMentions": [{"text": "梁某某", "label": "人名"}, {"text": "冰毒", "label": "毒品"}, {"text": "1.25克", "label": "毒品重量"}, {"text": "上海市浦东新区川沙新镇中医医院", "label": "地名"}, {"text": "2014年1月11日9时", "label": "时间"}], "sentText": "经查，梁某某于2014年1月11日9时在上海市浦东新区川沙新镇中医医院被警方抓获，身上查获冰毒1.25克。"}]}


### 自动化程序

In [ ]:
import json
import random
from zhipuai import ZhipuAI
from tqdm import tqdm
# 初始化 ZhipuAI 客户端
client = ZhipuAI(api_key="7bb97f4973df63dbb95018423009ed65.g4p3Q4YfZgdtOm4s")  # 请替换为你的 API Key

# 加载 categorized_data.json 和 train.json
with open('categorized_data.json', 'r', encoding='utf-8') as f:
    categorized_data = json.load(f)

with open('train.json', 'r', encoding='utf-8') as f:
    train_data_lines = f.readlines()

# 定义函数：随机采样实体
def sample_entities(label, max_samples=3):
    entities = categorized_data.get(label, [])
    sample_count = random.randint(1, max_samples)
    sampled_entities = random.sample(entities, min(sample_count, len(entities)))
    return [entity['text'] for entity in sampled_entities]

# 定义实体类型和映射
required_labels = ['Nh', 'NDR']
optional_labels = ['Ns', 'NT', 'NW']
key_mapping = {
    'Nh': '人名',
    'NDR': '毒品',
    'NT': '时间',
    'Ns': '地名',
    'NW': '毒品重量'
}

# 定义函数：生成 Prompt
def generate_prompt(entities, example_sentences):
    prompt = "根据以下提供的实体类型和具体实体，生成一个包含相关内容的句子，类似于给定的样本句子风格。生成的句子需要真实且符合上下文逻辑，并包括以下实体类型中至少 3 种的内容：\n\n"
    
    # 添加实体类型和实体数据
    prompt += "### 实体类型与实体：\n"
    for entity_type, values in entities.items():
        prompt += f"- **{entity_type}**：{', '.join(values)}\n"
    
    # 添加示例句子
    prompt += "\n### 示例句子风格：\n"
    for sentence in example_sentences:
        prompt += f"- \"{sentence}\"\n"
    
    prompt += """\n### 生成要求：\n请生成一个句子，要求逻辑清晰且至少包含以下实体类型中的人名和毒品实体。并且请按照这种格式输出：{\"entityMentions\": [{\"text\": \"林某某\", \"label\": \"人名\"}, {\"text\": \"海洛因\", \"label\": \"毒品\"}], \"sentText\": \"经查，林某某因贩卖海洛因被警方逮捕\"}"""
    return prompt

# 循环 500 次并保存结果
results = []
for i in tqdm(range(500),desc="processing"):
    # 随机采样实体
    sampled_data = {
        'Nh': sample_entities('Nh', max_samples=3),
        'NDR': sample_entities('NDR', max_samples=3),
    }
    chosen_optional_labels = random.sample(optional_labels, random.randint(1, len(optional_labels)))
    for label in chosen_optional_labels:
        sampled_data[label] = sample_entities(label, max_samples=3)
    
    # 映射为中文键
    mapped_data = {key_mapping.get(k, k): v for k, v in sampled_data.items()}
    #mapped_data=sampled_data
    # 随机选取示例句子
    random.shuffle(train_data_lines)
    matching_sentences = []
    for line in train_data_lines:
        data = json.loads(line)
        sent_text = data.get("sentText", "")
        entity_mentions = data.get("entityMentions", [])
        entity_texts = {entity['text'] for entity in entity_mentions}
        if any(entity in entity_texts for entity_list in sampled_data.values() for entity in entity_list):
            matching_sentences.append(sent_text)
        if len(matching_sentences) >= 3:
            break

    # 生成 Prompt
    prompt_text = generate_prompt(mapped_data, matching_sentences)
    
    # 调用 ZhipuAI API
    try:
        response = client.chat.completions.create(
            model="glm-4-flash",  # 请填写您要调用的模型名称
            messages=[
                {"role": "user", "content": prompt_text},
            ],
        )
    except:
        continue
        
    # 将结果保存
    result = {
        "response": response.choices[0].message.content
    }
    results.append(result)
    
    # 打印进度
    #print(f"Iteration {i + 1}/500 completed.")

# 保存结果到 JSON 文件
with open('api_results1.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

print("All iterations completed. Results saved to 'api_results1.json'.")

processing: 100%|██████████| 500/500 [22:45<00:00,  2.73s/it]

All iterations completed. Results saved to 'api_results1.json'.


In [19]:
import json


def clean_responses(data_list):
    cleaned_list = []
    
    for item in data_list:
        # 1. 取出 response 字段中的字符串
        response_str = item["response"]

        # 2. 去掉前缀和后缀的 ```json\n 或 ```（如果有）
        #    同时去除所有换行符 \n
        response_str = response_str.replace('```json\n', '').replace('```', '').replace('\n', '').replace('\\','')

        # 3. 将清洗之后的字符串解析成 Python dict
        #    （如果本身就是一个 JSON，则可以成功解析）
        obj = json.loads(response_str)

        # 4. 构造你想要的最终输出格式：
        #    {
        #      "entityMentions":[{'text': 'xxx', 'label': 'xxx'}, ...],
        #      "sentText":"..."
        #    }
        #    注意：这里把 entityMentions 每个元素的 key 用单引号，最外层保持 JSON 风格。
        
        # 先处理 entityMentions
        entity_mentions_list = []
        for em in obj["entityMentions"]:
            # 用单引号包裹 'text' 和 'label'
            entity_mentions_list.append(f"{{'text': '{em['text']}', 'label': '{em['label']}'}}")
        
        # 拼成字符串
        entity_mentions_str = "[" + ", ".join(entity_mentions_list) + "]"

        # 再拼接整个结果
        # 注意最外层的 key 依然用双引号，保证它在 JSON（或类 JSON）里合法可读
        final_str = (
            f'{{'
            f'"entityMentions":{entity_mentions_str},'
            f'"sentText":"{obj["sentText"]}"'
            f'}}'
        )

        cleaned_list.append(final_str)
    
    return cleaned_list

if __name__ == "__main__":
    with open('/Users/yuu/Downloads/毕业设计/spider_bysj/api_results1.json', 'r', encoding='utf-8') as f:
        data = json.load(f)
    result = clean_responses(data)
    with open("/Users/yuu/Downloads/毕业设计/spider_bysj/api_results1.json", "w", encoding="utf-8") as f:
        json.dump(result, f ,ensure_ascii=False, indent=4)

In [22]:
import json

with open('/Users/yuu/Downloads/毕业设计/spider_bysj/api_results1.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# 解析数据并移除外层双引号
cleaned_data = []
for item in data:
    # 将外层双引号解析为 JSON 对象
    parsed_item = json.loads(item.replace("'", "\""))  # 这里将内部的单引号替换为双引号以符合 JSON 标准
    cleaned_data.append(parsed_item)

with open("/Users/yuu/Downloads/毕业设计/spider_bysj/api_results1.json", "w", encoding="utf-8") as f:
        json.dump(cleaned_data, f ,ensure_ascii=False, indent=4)

In [23]:
import json

def add_offsets_to_entities(json_file_path, output_file_path=None):
    """
    读取 JSON 文件，针对每个实体，在 sentText 中找出其 start/end 位置信息。
    最终将实体更新为形如：
        {"start": ..., "end": ..., "text": ..., "label": ...}
    """

    # 1. 读取 JSON 文件
    with open(json_file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)  # data 是一个 list，每个元素是一个 dict
    
    # 2. 遍历 data
    for item in data:
        sent = item["sentText"]          # 完整的句子
        new_mentions = []                # 用于存储带有 start/end 的实体

        for mention in item["entityMentions"]:
            mention_text = mention["text"]
            label = mention["label"]

            # 3. 查找 mention_text 在 sent 中的起始下标
            start_idx = sent.find(mention_text)

            # 如果查不到则根据需求进行处理，这里简单跳过
            if start_idx == -1:
                continue

            end_idx = start_idx + len(mention_text)  # end 为 [start, end) 范围的闭区间上限

            # 4. 构造新的实体格式
            new_mention = {
                "start": start_idx,
                "end": end_idx,
                "text": mention_text,
                "label": label
            }

            new_mentions.append(new_mention)
        
        # 将带有位置信息的实体列表回写到 item 中
        item["entityMentions"] = new_mentions

    # 5. 如果需要写回到新文件，可以执行以下操作
    if output_file_path:
        with open(output_file_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
    else:
        # 如果不需要写回，可直接返回处理后的结果
        return data


# ========== 示例调用方式 ==========
if __name__ == "__main__":
    input_path = '/Users/yuu/Downloads/毕业设计/spider_bysj/api_results1.json'
    output_path = '/Users/yuu/Downloads/毕业设计/spider_bysj/api_results2.json'

    # 将处理结果输出到新文件
    add_offsets_to_entities(input_path, output_file_path=output_path)

    # 或者直接得到返回值在内存中使用：
    # processed_data = add_offsets_to_entities(input_path)
    # print(processed_data)

## 关系推理

In [27]:
import requests
import json
import base64

def gpt_4_call(text, api_key, url="https://api.chatanywhere.tech/v1/chat/completions"):

    payload = json.dumps({
    "model": "gpt-4o-2024-11-20",
    "temperature": 1.0,
    "messages": [
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            "content": text
        }
    ]
    })
    headers = {
    'Authorization': f'Bearer {api_key}',
    'Content-Type': 'application/json'
    }

    response = requests.request("POST", url, headers=headers, data=payload)

    if response.status_code == 200:
        try:
            result = response.json()
            return result["choices"][0]["message"]["content"]
        except KeyError:
            return "Unexpected response format."
    else:
        return f"Error: {response.status_code} - {response.text}"

def gpt_4_turbo_call(image_path, text, api_key, url="https://api.chatanywhere.tech/v1/chat/completions"):
    # 将图像文件转换为 Base64
    def jpg_to_base64(image_path):
        with open(image_path, "rb") as image_file:
            binary_data = image_file.read()
            base64_data = base64.b64encode(binary_data).decode('utf-8')
            return f"data:image/jpeg;base64,{base64_data}"

    # 准备请求数据
    base64_string = jpg_to_base64(image_path)
    payload = json.dumps({
        "model": "gpt-4o-2024-11-20",
        "temperature": 1.0,
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": text,
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": base64_string
                        },
                    },
                ],
            }
        ]
    })
    headers = {
        'Authorization': f'Bearer {api_key}',
        'Content-Type': 'application/json'
    }

    # 发出 POST 请求
    response = requests.post(url, headers=headers, data=payload)

    # 解析并返回结果
    if response.status_code == 200:
        try:
            result = response.json()
            #print(result)
            return result["choices"][0]["message"]["content"]
        except KeyError:
            return "Unexpected response format."
    else:
        return f"Error: {response.status_code} - {response.text}"
    
api_key = "sk-CznQFqe2BPAlAn7DsfNPL3sceXVVS6Fp88NxjbH7b6OBcWis"  # 替换为实际的 API 密钥

In [39]:
import json
from tqdm import tqdm
input_path = "/Users/yuu/Downloads/毕业设计/spider_bysj/api_results2.json"   # 输入 JSON 文件
output_path = "/Users/yuu/Downloads/毕业设计/spider_bysj/relations2.json"
with open(input_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

results = []

# 2. 针对 JSON 文件中的每条记录构造 Prompt
for idx, record in tqdm(enumerate(data)):
    sent_text = record["sentText"]
    entity_mentions = record["entityMentions"]

    # 3. 构造可读的实体列表信息
    #    例如："张某乙(人名), 华山路(地名), 冰毒(毒品)"
    entity_descriptions = [
        f"{em['text']}({em['label']})" 
        for em in entity_mentions
    ]
    entities_str = ", ".join(entity_descriptions)

    # 4. 根据业务需求，自定义 prompt 结构和引导语
    #    此处给出一个示例：
    prompt = f"""现在定义四种关系，分别是1.贩卖毒品 2.贩卖给人 3.持有毒品 4.非法容留他人吸毒。请阅读以下句子和实体信息，推理可能的关系，并严格输出 JSON 格式：
句子: "{sent_text}"
实体: {entities_str}

输出格式示例：
{{
"relationMentions":[
{{ "em1Text": "纳某某", "label": "贩卖毒品", "em2Text": "冰毒" }},
{{ "em1Text": "纳某某", "label": "贩卖给人", "em2Text": "邓某" }}
]
}}

请根据句子含义，生成类似上述的 "relationMentions" 列表（如无关系可留空），并保证只输出 JSON。
"""
    # 收集 prompt
    result = {
      "relationMentions": gpt_4_call(prompt,api_key)
    } 
    results.append(result)
with open(output_path, 'w', encoding='utf-8') as f:
  json.dump(results, f, ensure_ascii=False, indent=4)

499it [14:44,  1.77s/it]


In [40]:
import json

def clean_relation_mentions(input_path, output_path):
    # 1. 读取 JSON 文件
    with open(input_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # 2. 遍历每条记录，去除不需要的字符
    for item in data:
        relation_str = item["relationMentions"]
        
        # 依次去除不需要的字符串
        # 也可以考虑用正则批量替换
        relation_str = relation_str.replace("```json\n", "")
        relation_str = relation_str.replace("```json", "")
        relation_str = relation_str.replace("```", "")
        relation_str = relation_str.replace("\\", "")  # 若想去掉所有反斜杠
        relation_str = relation_str.replace("\n", "")  # 去掉换行符

        item["relationMentions"] = relation_str

    # 3. 将清洗后的数据写回文件（或另外保存到新文件）
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

if __name__ == "__main__":
    input_file = '/Users/yuu/Downloads/毕业设计/spider_bysj/relations2.json'
    output_file = '/Users/yuu/Downloads/毕业设计/spider_bysj/relations2_1.json'
    clean_relation_mentions(input_file, output_file)
    print("清洗完成！")

清洗完成！


In [41]:
import json

def clean_relation_mentions(input_path, output_path):
    # 1. 读取 JSON 文件
    with open(input_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # 2. 遍历每条记录，清洗 relationMentions 的值
    for item in data:
        relation_value = item["relationMentions"]
        
        # 去掉前后的双引号（若存在）
        if relation_value.startswith('"') and relation_value.endswith('"'):
            relation_value = relation_value[1:-1]
        
        # 去掉反斜杠
        relation_value = relation_value.replace("\\", "")
        
        # 更新清洗后的值
        item["relationMentions"] = relation_value

    # 3. 将清洗后的数据写回文件（或保存为新文件）
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

if __name__ == "__main__":
    input_file = '/Users/yuu/Downloads/毕业设计/spider_bysj/relations2_1.json'
    output_file = '/Users/yuu/Downloads/毕业设计/spider_bysj/relations2_1.json'
    clean_relation_mentions(input_file, output_file)
    print("清洗完成！")

清洗完成！


In [42]:
import json

# 读取 JSON 文件
with open('/Users/yuu/Downloads/毕业设计/spider_bysj/relations2_1.json', 'r', encoding='utf-8') as file:
    data = json.load(file)

# 处理每一条记录
for item in data:
    if 'relationMentions' in item:
        # 获取原始值并解析为 JSON 对象
        raw_value = item['relationMentions']
        # 去掉前后的引号和反斜杠，解析为 Python 字典
        cleaned_value = json.loads(raw_value.strip('"').replace('\\', ''))
        item['relationMentions'] = cleaned_value

# 保存结果到新的 JSON 文件
with open('/Users/yuu/Downloads/毕业设计/spider_bysj/relations2_1.json', 'w', encoding='utf-8') as file:
    json.dump(data, file, ensure_ascii=False, indent=4)

print("清洗完成，结果已保存到 output.json")

清洗完成，结果已保存到 output.json


In [53]:
import json

# 读取 JSON 文件
with open('/Users/yuu/Downloads/毕业设计/spider_bysj/relations2_1.json', 'r', encoding='utf-8') as file:
    data = json.load(file)

# 处理每条记录
for item in data:
    if "relationMentions" in item and "relationMentions" in item["relationMentions"]:
        # 将嵌套的 relationMentions 替换为内部的 relationMentions
        item["relationMentions"] = item["relationMentions"]["relationMentions"]

# 保存结果到新的 JSON 文件
with open('/Users/yuu/Downloads/毕业设计/spider_bysj/relations2_1.json', 'w', encoding='utf-8') as file:
    json.dump(data, file, ensure_ascii=False, indent=4)

print("处理完成，结果已保存到 output.json")

处理完成，结果已保存到 output.json


## 把关系拼接到实体里面

In [ ]:
with open('/Users/yuu/Downloads/毕业设计/spider_bysj/relations2_1.json', 'r', encoding='utf-8') as file:
    relation_data = json.load(file)

with open('/Users/yuu/Downloads/毕业设计/spider_bysj/api_results2.json', 'r', encoding='utf-8') as file:
    entity_data = json.load(file)

for entity_item, relation_item in zip(entity_data, relation_data):
    # 检查关系数据中是否有 "relationMentions"
    if "relationMentions" in relation_item:
        # 将 "relationMentions" 拼接到实体数据中
        entity_item["relationMentions"] = relation_item["relationMentions"]

# 保存结果到新的 JSON 文件
with open('merged.json', 'w', encoding='utf-8') as file:
    json.dump(entity_data, file, ensure_ascii=False, indent=4)

In [70]:
with open('/Users/yuu/Downloads/毕业设计/spider_bysj/output.json', 'r', encoding='utf-8') as file:
    data = json.load(file)

# 将每个对象单独存储到文件，每行一个 JSON 对象
with open('/Users/yuu/Downloads/毕业设计/spider_bysj/output1.json', 'w', encoding='utf-8') as file:
    for obj in data:
        json.dump(obj, file, ensure_ascii=False)
        file.write('\n')  # 每个对象写在单独的一行

In [69]:
import json
import time
import random
from zhipuai import ZhipuAI
from tqdm import tqdm
# 初始化 ZhipuAI 客户端
client = ZhipuAI(api_key="7bb97f4973df63dbb95018423009ed65.g4p3Q4YfZgdtOm4s")  # 请替换为你的 API Key

# 这里假设你已经在其他地方完成了 client 的初始化，比如：
# import openai
# openai.api_key = "your_api_key"
# client = openai  # 或者你自己的 client 封装
# 下文示例中直接使用 client 进行演示

def call_llm_for_replacement(data_item: dict, client) -> dict:
    """
    遍历 data_item 中的 entityMentions，对每个实体 text 调用大模型进行替换，
    并将替换结果同步更新到 entityMentions、sentText、relationMentions。
    
    :param data_item: 原始 JSON 中的单条记录(字典)，包含 entityMentions、sentText、relationMentions 等字段
    :param client:    你已创建好的大模型 client，用于调用 API
    :return:          完成替换后更新的 data_item
    """

    # 如果没有 entityMentions，直接返回
    if "entityMentions" not in data_item:
        return data_item

    # 遍历当前记录里的每个实体
    for entity in data_item["entityMentions"]:
        old_text = entity["text"]
        
        # 1) 构建给大模型的提示语
        #    这里的 prompt_text 仅是示例，你可根据实际业务编写：
        prompt_text = (
            f"请将以下实体名称替换为另一个同类型的名称，但保证满足现实性或一致性需求。\n"
            f"实体: {old_text}。\n"
            f"只需要返回修改过后的实体即可，请确保不包含其他任何的前缀或者标点符号等信息。"
        )
        
        # 2) 调用大模型 API
        try:
            response = client.chat.completions.create(
                model="glm-4-flash",  # 你的模型名称，如 "glm-4-flash"
                messages=[{"role": "user", "content": prompt_text}],
            )
            # 假设返回结果在 response["choices"][0]["message"]["content"] 中
            new_text = response.choices[0].message.content.strip()
            #print(new_text)
        except Exception as e:
            print(f"[WARNING] 调用API失败，保留原实体: {old_text}，错误信息: {e}")
            new_text = old_text
        
        # 如果大模型返回的文本是空或无效，也可以做一些保护性逻辑
        if not new_text:
            new_text = old_text
        
        # 3) 将 old_text 替换为 new_text，并同步更新各处
        entity["text"] = new_text  # 更新 entityMentions

        # 更新 sentText 中所有出现该实体 old_text 的地方
        if "sentText" in data_item and old_text in data_item["sentText"]:
            data_item["sentText"] = data_item["sentText"].replace(old_text, new_text)

        # 更新 relationMentions 中出现该实体 old_text 的地方
        if "relationMentions" in data_item:
            for relation in data_item["relationMentions"]:
                if relation["em1Text"] == old_text:
                    relation["em1Text"] = new_text
                if relation["em2Text"] == old_text:
                    relation["em2Text"] = new_text
        
        # 如果你的接口调用频率有要求，可以在此 sleep 一下，防止触发频率限制
        # time.sleep(0.5)

    return data_item


def main():
    data_list = []
    with open('generate_train.json', 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            data = json.loads(line)
            data_list.append(data)

    # data_list 现在是 [dict1, dict2, ...] 的格式，每行一个 dict

    new_data = []
    for item in tqdm(data_list,desc='Processing'):
        # 针对每个 JSON 对象 item，调用 LLM 进行实体替换
        new_item = call_llm_for_replacement(item, client)
        new_data.append(new_item)

    # 将处理后的所有对象组合成一个列表写回文件
    with open("output.json", "w", encoding="utf-8") as f:
        json.dump(new_data, f, ensure_ascii=False, indent=4)

    print("实体替换完成，结果已保存到 output.json")



if __name__ == "__main__":

    main()

Processing:  13%|█▎        | 244/1914 [20:26<1:59:26,  4.29s/it]

[WARNING] 调用API失败，保留原实体: 甲卡西酮，错误信息: Error code: 400, with error text {"contentFilter":[{"level":2,"role":"assistant"}],"error":{"code":"1301","message":"系统检测到输入或生成内容可能包含不安全或敏感内容，请您避免输入易产生敏感内容的提示语，感谢您的配合。"}}


Processing:  13%|█▎        | 257/1914 [21:34<2:22:33,  5.16s/it]

[WARNING] 调用API失败，保留原实体: 中华人民共和国，错误信息: Error code: 400, with error text {"contentFilter":[{"level":1,"role":"assistant"}],"error":{"code":"1301","message":"系统检测到输入或生成内容可能包含不安全或敏感内容，请您避免输入易产生敏感内容的提示语，感谢您的配合。"}}


Processing:  15%|█▍        | 281/1914 [23:23<2:05:00,  4.59s/it]

...

[WARNING] 调用API失败，保留原实体: 亚甲基二氧甲基苯丙胺，错误信息: Error code: 400, with error text {"contentFilter":[{"level":2,"role":"assistant"}],"error":{"code":"1301","message":"系统检测到输入或生成内容可能包含不安全或敏感内容，请您避免输入易产生敏感内容的提示语，感谢您的配合。"}}


Processing:  87%|████████▋ | 1669/1914 [2:04:13<04:03,  1.01it/s]  

[WARNING] 调用API失败，保留原实体: 亚甲基二氧甲基苯丙胺，错误信息: Error code: 400, with error text {"contentFilter":[{"level":2,"role":"assistant"}],"error":{"code":"1301","message":"系统检测到输入或生成内容可能包含不安全或敏感内容，请您避免输入易产生敏感内容的提示语，感谢您的配合。"}}


Processing: 100%|██████████| 1914/1914 [2:09:40<00:00,  4.06s/it]


实体替换完成，结果已保存到 output.json


In [72]:
import json

def filter_json_file(input_file, output_file):
    """
    过滤 JSON 文件，删除包含指定字符（如“修改”或“→”）的行。
    :param input_file: 输入 JSON 文件路径
    :param output_file: 输出过滤后的 JSON 文件路径
    """
    # 定义需要过滤的关键字
    keywords = ["修改", "->"]

    # 打开输入文件逐行读取
    with open(input_file, 'r', encoding='utf-8') as infile, open(output_file, 'w', encoding='utf-8') as outfile:
        for line in infile:
            # 如果行中包含任何关键字，跳过
            if any(keyword in line for keyword in keywords):
                continue
            # 写入不包含关键字的行
            outfile.write(line)

    print(f"过滤完成，结果已保存到 {output_file}")

# 调用过滤函数
filter_json_file('/Users/yuu/Downloads/毕业设计/spider_bysj/实体替换原始数据.json', '/Users/yuu/Downloads/毕业设计/spider_bysj/filtered_output.json')

过滤完成，结果已保存到 /Users/yuu/Downloads/毕业设计/spider_bysj/filtered_output.json
